# Velocity-model source workbench

This notebook preserves the reusable **source snapshots** from the teammate's GRU project. It deliberately excludes raw data, generated windows, results, plots, model weights, and the Random-Forest baseline.

The snapshots are reference material, not production-ready notebook cells. Do **not** run them unchanged: they take raw phone-frame acceleration (including gravity) and gyroscope readings at 10 Hz, whereas IDR will provide synchronized, calibrated, vehicle-frame, gravity-removed inputs.

Use the code here to refactor the training pipeline after the deterministic sensor stage is complete. Keep the original teammate project untouched as provenance.

## Confirmed current artifacts

- **GRU checkpoint:** 6 features, 50 samples (5 s at 10 Hz), 2 GRU layers with hidden size 64, dropout 0.2, then a 64 → 32 → 1 head.
- **GRU held-out sequence result:** MAE 17.24 km/h, RMSE 23.02 km/h, R² 0.267. This is a baseline, not deployment-ready.
- **CNN checkpoint:** 7 features, 50 samples, channels (32, 64, 64), kernel size 3, dropout 0.3. No matching source code, scalers, manifest, or reported evaluation was supplied, so it cannot yet be compared or used safely.
- Neither standalone checkpoint has its feature scaler or target scaler. They cannot produce physical-speed values by themselves.

In [ ]:
# This cell is safe to run. It locates the unchanged teammate source project.
from pathlib import Path

WORKSPACE_ROOT = Path.cwd().resolve()
TEAMMATE_PROJECT = (
    WORKSPACE_ROOT
    / "SIH-2-main"
    / "SIH-2-main"
    / "IOVNBD-Speed-Prediction"
)
TEAMMATE_SOURCE = TEAMMATE_PROJECT / "src"

if not TEAMMATE_SOURCE.is_dir():
    raise FileNotFoundError(
        "Open this notebook with the workspace root as the working directory, "
        "or update WORKSPACE_ROOT."
    )

print(f"Reusable source project: {TEAMMATE_PROJECT}")

## Required refactor before IDR training

1. Build training features from the deterministic pipeline: synchronized IMU → orientation → phone-to-vehicle calibration → gravity removal.
2. Decide and record the final feature order, sampling rate, and causal window length; do not inherit the current six or seven features blindly.
3. Train the target in **m/s**, not km/h, so the velocity observation matches the EKF state.
4. Fit and ship feature/target scalers with every checkpoint plus a JSON manifest containing feature order, units, frame, sample rate, window length, model version, and training-data split.
5. Hold out whole journeys/drivers/phones/vehicles. The current GRU does hold out whole named sequences, which is a useful starting pattern.
6. Export a velocity adapter only after the model passes this revised evaluation; uncertainty estimation will consume its speed prediction afterward.

## Teammate source snapshots

Each following cell is a faithful source copy apart from its command-line auto-run guard, which is disabled so a notebook execution cannot accidentally preprocess 19+ hours of data or start training. They use the teammate project's imports and paths, so treat them as editable reference code for the later backend refactor.

### `config.py`

In [ ]:
# SOURCE SNAPSHOT — config.py
# Do not run unchanged. Copy/refactor into the IDR velocity-training work when its input contract is finalized.

"""
Central configuration for the IO-VNBD speed-prediction pipeline.

All paths are relative to the project root so the project runs unchanged
on any machine (Windows/Linux/Mac), as long as it's launched from the
project root (e.g. `python run_pipeline.py` from IOVNBD-Speed-Prediction/).
"""
import os
import glob

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")

MODELS_DIR = os.path.join(ROOT_DIR, "models")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "artifacts")
RESULTS_DIR = os.path.join(ROOT_DIR, "results")
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")


def discover_sequence_pairs():
    """Auto-discover all matched S-<name>.csv / V-<name>.csv pairs under
    data/raw/. Returns a sorted list of (sequence_name, s_path, v_path)
    tuples. A sequence is only included if BOTH its S- and V- file exist.
    This lets you add more driving sequences just by dropping matching
    files into data/raw/ - no code changes needed beyond re-running the
    pipeline."""
    pairs = []
    for s_path in sorted(glob.glob(os.path.join(RAW_DATA_DIR, "S-*.csv"))):
        name = os.path.basename(s_path)[len("S-"):-len(".csv")]
        v_path = os.path.join(RAW_DATA_DIR, f"V-{name}.csv")
        if os.path.exists(v_path):
            pairs.append((name, s_path, v_path))
        else:
            print(f"[config] WARNING: found {s_path} but no matching V-{name}.csv - skipping this sequence.")
    return pairs

# ---------------------------------------------------------------------------
# Column mapping (verified by actually inspecting S-M.csv / V-M.csv headers)
# ---------------------------------------------------------------------------
# Smartphone (S) file columns of interest
S_COLS = {
    "acc_x": "ACCELEROMETER X (m/s²)",
    "acc_y": "ACCELEROMETER Y (m/s²)",
    "acc_z": "ACCELEROMETER Z (m/s²)",
    # The S-file's gyroscope columns are labelled by rotation axis name
    # (Yaw/Pitch/Roll) rather than X/Y/Z. We map them onto x/y/z using the
    # conventional aerospace correspondence roll->x, pitch->y, yaw->z.
    # This is a naming-convention assumption (the raw values themselves are
    # not altered) and is documented here for transparency.
    "gyro_x": "GYROSCOPE Roll (rad/s)",
    "gyro_y": "GYROSCOPE Pitch (rad/s)",
    "gyro_z": "GYROSCOPE Yaw (rad/s)",
    "date": "DATE (YYYY-MO-DD HH-MI-SS_SSS)",
    # Present but NOT used as model inputs/targets in this stage:
    "gps_speed": "GPS SPEED (Kmh)",
    "gps_lat": "GPS LATITUDE (degrees)",
    "gps_lon": "GPS LONGITUDE (degrees)",
    "time_since_start_ms": "TIME SINCE START (ms)",  # unreliable: resets mid-file, do not use for timing
}

# Vehicle/reference (V) file columns of interest
V_COLS = {
    # Non-GPS, CAN-bus/ECU-derived reference vehicle speed. Chosen as the
    # prediction target because it is NOT derived from GPS (unlike
    # "Velocity (km/hr)", which comes from the GPS-based VBOX unit).
    "vehicle_speed": "Indicated Vehicle Speed (km/hr)",
    "time_since_start_s": "Time Since Start of Day (seconds)",
    # Present but NOT used in this stage:
    "gps_velocity": "Velocity (km/hr)",
}

DATE_FORMAT = "%Y-%m-%d %H:%M:%S:%f"

# ---------------------------------------------------------------------------
# Standardized dataframe column order (after preprocessing)
# ---------------------------------------------------------------------------
STANDARD_COLUMNS = [
    "sequence_id",
    "timestamp",
    "acc_x", "acc_y", "acc_z",
    "gyro_x", "gyro_y", "gyro_z",
    "vehicle_speed",
]

FEATURE_COLUMNS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
TARGET_COLUMN = "vehicle_speed"

# ---------------------------------------------------------------------------
# Sampling / windowing (determined from actual data inspection: both S and V
# files are sampled at 10 Hz -> sample period 0.1 s)
# ---------------------------------------------------------------------------
SAMPLING_RATE_HZ = 10.0
SAMPLE_PERIOD_S = 1.0 / SAMPLING_RATE_HZ

WINDOW_DURATION_S = 5.0
WINDOW_SIZE = int(round(WINDOW_DURATION_S * SAMPLING_RATE_HZ))  # 20 timesteps

# Stride between successive windows, in samples. A stride > 1 reduces the
# number of near-duplicate overlapping windows (helps generalization and
# keeps training lightweight on CPU) while still using every part of the
# sequence.
WINDOW_STRIDE = 5  # 0.5 s between window starts

# ---------------------------------------------------------------------------
# Chronological split fractions (train / val / test), applied to the single
# available sequence in time order to avoid leakage.
# ---------------------------------------------------------------------------
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

RANDOM_SEED = 42


### `inspect_data.py`

In [ ]:
# SOURCE SNAPSHOT — inspect_data.py
# Do not run unchanged. Copy/refactor into the IDR velocity-training work when its input contract is finalized.

"""
Step 1: Dataset inspection.

Auto-discovers every matched S-<name>.csv / V-<name>.csv pair under
data/raw/ and inspects each one (structure, sampling rate, duration,
missing values, duplicate timestamps, speed range).

Run directly:
    python src/inspect_data.py
"""
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as cfg


def load_raw_pair(s_path: str, v_path: str):
    s = pd.read_csv(s_path, encoding="latin1")
    v = pd.read_csv(v_path, encoding="latin1")
    s.columns = [c.strip() for c in s.columns]
    v.columns = [c.strip() for c in v.columns]
    return s, v


def load_raw():
    """Backward-compatible helper: loads the FIRST discovered sequence
    pair. Prefer load_all_raw() for multi-sequence use."""
    pairs = cfg.discover_sequence_pairs()
    if not pairs:
        raise FileNotFoundError(
            f"No matched S-*.csv / V-*.csv pairs found in {cfg.RAW_DATA_DIR}.\n"
            f"Place real (non-Git-LFS-pointer) files there, e.g. S-M.csv + V-M.csv."
        )
    name, s_path, v_path = pairs[0]
    return load_raw_pair(s_path, v_path)


def load_all_raw():
    """Loads every discovered sequence pair. Returns a dict:
    {sequence_name: (s_df, v_df)}"""
    pairs = cfg.discover_sequence_pairs()
    if not pairs:
        raise FileNotFoundError(
            f"No matched S-*.csv / V-*.csv pairs found in {cfg.RAW_DATA_DIR}.\n"
            f"Place real (non-Git-LFS-pointer) files there, e.g. S-M.csv + V-M.csv."
        )
    out = {}
    for name, s_path, v_path in pairs:
        out[name] = load_raw_pair(s_path, v_path)
    return out


def _inspect_one(name, s, v):
    print("\n" + "-" * 70)
    print(f"SEQUENCE: {name}")
    print("-" * 70)
    print(f"  S-file shape   : {s.shape}")
    print(f"  V-file shape   : {v.shape}")

    same_len = len(s) == len(v)
    print(f"  Row counts match (S == V): {same_len} ({len(s)} vs {len(v)})")

    print("  Missing values (S):", int(s.isna().sum().sum()))
    print("  Missing values (V):", int(v.isna().sum().sum()))

    s_num = s.select_dtypes(include=[np.number])
    v_num = v.select_dtypes(include=[np.number])
    print("  Infinite values (S):", int(np.isinf(s_num).sum().sum()))
    print("  Infinite values (V):", int(np.isinf(v_num).sum().sum()))

    dt = pd.to_datetime(s[cfg.S_COLS["date"]], format=cfg.DATE_FORMAT)
    dup_dates = int(dt.duplicated().sum())
    diffs = dt.diff().dt.total_seconds().dropna()
    duration_s = (dt.iloc[-1] - dt.iloc[0]).total_seconds()

    print(f"  Duration (s)   : {duration_s:.1f}  (~{duration_s/60:.1f} min)")
    print(f"  Duplicate dates: {dup_dates}")
    if len(diffs):
        print(f"  Median sample period (s): {diffs.median():.4f}  "
              f"(~{1.0/diffs.median():.2f} Hz)")
        print(f"  Gaps > 0.5s    : {int((diffs > 0.5).sum())}")

    speed = v[cfg.V_COLS["vehicle_speed"]]
    print(f"  Speed range (km/h): min={speed.min():.2f} max={speed.max():.2f} mean={speed.mean():.2f}")
    print(f"  % near-zero speed (<1 km/h): {(speed < 1).mean()*100:.1f}%")

    return {
        "name": name,
        "n_rows": len(s),
        "duration_s": duration_s,
        "sampling_hz": 1.0 / diffs.median() if len(diffs) else float("nan"),
        "dup_dates": dup_dates,
        "missing_s": int(s.isna().sum().sum()),
        "missing_v": int(v.isna().sum().sum()),
        "speed_min": float(speed.min()),
        "speed_max": float(speed.max()),
        "speed_mean": float(speed.mean()),
    }


def inspect():
    all_raw = load_all_raw()

    print("=" * 70)
    print("DATASET INSPECTION REPORT")
    print("=" * 70)
    print(f"Discovered {len(all_raw)} sequence pair(s): {list(all_raw.keys())}")

    summaries = []
    for name, (s, v) in all_raw.items():
        summaries.append(_inspect_one(name, s, v))

    print("\n" + "=" * 70)
    print("SUMMARY ACROSS ALL SEQUENCES")
    print("=" * 70)
    summary_df = pd.DataFrame(summaries)
    print(summary_df.to_string(index=False))
    total_rows = summary_df["n_rows"].sum()
    total_duration_h = summary_df["duration_s"].sum() / 3600
    print(f"\nTotal rows across all sequences: {total_rows}")
    print(f"Total duration: {total_duration_h:.2f} hours")

    # -- column mapping table (same mapping applies to every sequence) -----
    print("\n" + "=" * 70)
    print("COLUMN MAPPING (applies to every sequence)")
    print("=" * 70)
    rows = [
        ("Timestamp", cfg.S_COLS["date"] + " (S-file)", "datetime, 10 Hz"),
        ("Accelerometer X", cfg.S_COLS["acc_x"], "m/s^2"),
        ("Accelerometer Y", cfg.S_COLS["acc_y"], "m/s^2"),
        ("Accelerometer Z", cfg.S_COLS["acc_z"], "m/s^2"),
        ("Gyroscope X (~Roll)", cfg.S_COLS["gyro_x"], "rad/s"),
        ("Gyroscope Y (~Pitch)", cfg.S_COLS["gyro_y"], "rad/s"),
        ("Gyroscope Z (~Yaw)", cfg.S_COLS["gyro_z"], "rad/s"),
        ("Vehicle speed (target)", cfg.V_COLS["vehicle_speed"] + " (V-file)", "km/h"),
        ("(unused) GPS speed", cfg.S_COLS["gps_speed"] + " (S-file)", "km/h"),
        ("(unused) GPS velocity", cfg.V_COLS["gps_velocity"] + " (V-file)", "km/h"),
    ]
    for purpose, colname, unit in rows:
        print(f"  {purpose:26s} | {colname:45s} | {unit}")

    return all_raw, summary_df


if False:  # Disabled in this source snapshot; call deliberately after adaptation.
    inspect()


### `preprocess.py`

In [ ]:
# SOURCE SNAPSHOT — preprocess.py
# Do not run unchanged. Copy/refactor into the IDR velocity-training work when its input contract is finalized.

"""
Step 2: Preprocessing.

Takes every discovered S-<name>.csv / V-<name>.csv pair and produces one
clean, standardized dataframe (all sequences concatenated) with columns:

    sequence_id, timestamp, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z, vehicle_speed

Each sequence is cleaned INDEPENDENTLY (its own timestamp parsing,
duplicate/NaN handling) before being concatenated, so no cleaning step
ever mixes rows across two different recordings.

Preprocessing steps (each with a stated reason), applied per sequence:

1. Parse timestamps from the S-file DATE column (the only reliable,
   monotonic timestamp source - see inspect_data.py notes).
2. Sort by timestamp (defensive; data was already monotonic on inspection,
   but we do not assume this holds for future data drops).
3. Row-align S and V. Every sequence pair was confirmed (inspect_data.py)
   to have matching row counts, consistent with them being the
   pre-synchronized "Synchronised V and S dataset" pairs. We therefore
   treat row index as the synchronization key within each sequence (row i
   of S == row i of V), matching how the dataset publishers describe this
   folder.
4. Drop rows with missing/NaN/infinite values in any feature or target
   column (guards against future data drops that may contain gaps).
5. Remove exact duplicate timestamps (guarded for, per sequence).
6. Unit verification: acceleration is already in m/s^2, gyroscope in
   rad/s, target speed in km/h - matching the mapping documented in
   config.py. No unit conversion needed.
7. Light sensor filtering: NONE applied by default (kept conservative, as
   instructed). A commented-out optional low-pass filter hook is provided
   for future experimentation but is OFF by default so raw dynamics are
   preserved for the model to learn from.
8. Feature scaling (StandardScaler) is fit ONLY on the training split's
   rows inside create_sequences.py (not here), to avoid any leakage from
   validation/test sequences into the training statistics.
"""
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as cfg
from inspect_data import load_all_raw


def _preprocess_one(name: str, s: pd.DataFrame, v: pd.DataFrame, apply_lowpass: bool = False) -> pd.DataFrame:
    timestamp = pd.to_datetime(s[cfg.S_COLS["date"]], format=cfg.DATE_FORMAT)

    df = pd.DataFrame({
        "sequence_id": name,
        "timestamp": timestamp,
        "acc_x": s[cfg.S_COLS["acc_x"]].astype(float),
        "acc_y": s[cfg.S_COLS["acc_y"]].astype(float),
        "acc_z": s[cfg.S_COLS["acc_z"]].astype(float),
        "gyro_x": s[cfg.S_COLS["gyro_x"]].astype(float),
        "gyro_y": s[cfg.S_COLS["gyro_y"]].astype(float),
        "gyro_z": s[cfg.S_COLS["gyro_z"]].astype(float),
        "vehicle_speed": v[cfg.V_COLS["vehicle_speed"]].astype(float),
    })

    n_before = len(df)

    df = df.sort_values("timestamp").reset_index(drop=True)

    n_dup = int(df["timestamp"].duplicated().sum())
    if n_dup > 0:
        df = df.drop_duplicates(subset="timestamp", keep="first").reset_index(drop=True)

    numeric_cols = [c for c in df.columns if c not in ("timestamp", "sequence_id")]
    n_nan = int(df[numeric_cols].isna().sum().sum())
    n_inf = int(np.isinf(df[numeric_cols]).sum().sum())
    if n_nan > 0 or n_inf > 0:
        df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
        df = df.dropna(subset=numeric_cols).reset_index(drop=True)

    n_after = len(df)

    if apply_lowpass:
        from scipy.signal import butter, filtfilt
        b, a = butter(N=2, Wn=3.0 / (cfg.SAMPLING_RATE_HZ / 2), btype="low")
        for c in ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]:
            df[c] = filtfilt(b, a, df[c].values)

    print(f"  [{name}] rows before={n_before}  dup_removed={n_dup}  "
          f"nan_inf_removed={n_before - n_dup - n_after}  rows after={n_after}")

    return df


def preprocess(apply_lowpass: bool = False) -> pd.DataFrame:
    all_raw = load_all_raw()

    print("=" * 70)
    print("PREPROCESSING SUMMARY (per sequence)")
    print("=" * 70)

    cleaned = []
    for name, (s, v) in all_raw.items():
        cleaned.append(_preprocess_one(name, s, v, apply_lowpass=apply_lowpass))

    df = pd.concat(cleaned, ignore_index=True)

    print(f"\nSequences processed : {len(cleaned)}")
    print(f"Total rows (all sequences): {len(df)}")
    print(f"Low-pass filter applied: {apply_lowpass}")
    print(f"Columns: {list(df.columns)}")

    os.makedirs(cfg.PROCESSED_DATA_DIR, exist_ok=True)
    out_path = os.path.join(cfg.PROCESSED_DATA_DIR, "clean_sequence.parquet")
    try:
        df.to_parquet(out_path, index=False)
    except Exception:
        out_path = os.path.join(cfg.PROCESSED_DATA_DIR, "clean_sequence.csv")
        df.to_csv(out_path, index=False)
    print(f"Saved cleaned dataframe -> {out_path}")

    return df


if False:  # Disabled in this source snapshot; call deliberately after adaptation.
    preprocess()


### `create_sequences.py`

In [ ]:
# SOURCE SNAPSHOT — create_sequences.py
# Do not run unchanged. Copy/refactor into the IDR velocity-training work when its input contract is finalized.

"""
Step 3: Training-data creation.

Task: smartphone IMU (6 channels) -> vehicle speed (1 value), using a
sliding window of past IMU samples ending at the prediction instant (no
future information is used).

Leakage-prevention strategy
----------------------------
**With 2+ sequences (recommended):** whole driving sequences are assigned
to train / val / test - never split within a sequence. This is the
strongest form of leakage prevention: it also protects against a model
that "memorizes" a specific route/driver/vehicle rather than learning a
general IMU -> speed relationship, since test sequences are entirely
unseen recordings. Sequences are greedily assigned to whichever split is
currently furthest below its target proportion (by row count), so the
achieved train/val/test sizes approximate 70/15/15 as closely as possible
given the available sequence sizes.

**With exactly 1 sequence (fallback):** there is no way to hold out a
whole unseen sequence, so we fall back to a chronological split of the
single continuous timeline: first 70% of rows -> train, next 15% -> val,
last 15% -> test. No shuffling of raw rows.

**In both cases:** windows are only created AFTER the split is decided,
and independently within each split's own row-ranges per sequence, so no
window ever mixes rows from two different splits or two different
sequences. The StandardScalers (features and target) are fit ONLY on the
training split's rows.

Window size
-----------
Sampling rate = 10 Hz (verified in inspect_data.py). We use a
WINDOW_DURATION_S = 2.0 s window (20 timesteps), which is long enough to
capture short-term acceleration/deceleration dynamics relevant to speed
but short enough to keep the model lightweight and responsive. Consecutive
windows are spaced WINDOW_STRIDE = 5 samples (0.5 s) apart to reduce
redundancy between neighboring windows.
"""
import os
import sys

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as cfg
from preprocess import preprocess


def _load_clean_df() -> pd.DataFrame:
    parquet_path = os.path.join(cfg.PROCESSED_DATA_DIR, "clean_sequence.parquet")
    csv_path = os.path.join(cfg.PROCESSED_DATA_DIR, "clean_sequence.csv")
    if os.path.exists(parquet_path):
        return pd.read_parquet(parquet_path)
    if os.path.exists(csv_path):
        return pd.read_csv(csv_path, parse_dates=["timestamp"])
    return preprocess()


def _assign_sequences_to_splits(df: pd.DataFrame):
    """Greedily assign whole sequences to train/val/test so the resulting
    row-count proportions approximate TRAIN/VAL/TEST_FRACTION as closely
    as possible. Returns a dict {split_name: [sequence_id, ...]}."""
    counts = df.groupby("sequence_id").size().sort_values(ascending=False)
    total = counts.sum()
    targets = {
        "train": cfg.TRAIN_FRACTION * total,
        "val": cfg.VAL_FRACTION * total,
        "test": cfg.TEST_FRACTION * total,
    }
    current = {"train": 0, "val": 0, "test": 0}
    assignment = {"train": [], "val": [], "test": []}

    for seq_id, n in counts.items():
        # assign to whichever split is currently furthest below its target
        deficits = {k: targets[k] - current[k] for k in current}
        best_split = max(deficits, key=deficits.get)
        assignment[best_split].append(seq_id)
        current[best_split] += n

    return assignment, current, total


def _make_windows(df_block: pd.DataFrame, window_size: int, stride: int):
    """Slide a window of `window_size` IMU rows within a SINGLE sequence's
    contiguous rows; the label is the vehicle speed at the LAST timestep
    of the window (i.e. "now"), so only past and present IMU samples are
    used - never future ones. Windows never cross a sequence boundary.

    Also returns, for each window, the timestamp and sequence_id at the
    window's last timestep, so downstream plotting/evaluation can align
    predictions back to real time without re-deriving split boundaries."""
    all_X, all_y, all_t, all_seq = [], [], [], []
    for seq_id, sub in df_block.groupby("sequence_id", sort=False):
        sub = sub.sort_values("timestamp")
        feats = sub[cfg.FEATURE_COLUMNS].values.astype(np.float32)
        target = sub[cfg.TARGET_COLUMN].values.astype(np.float32)
        times = sub["timestamp"].values
        n = len(sub)
        starts = range(0, n - window_size + 1, stride)
        for i in starts:
            all_X.append(feats[i:i + window_size])
            all_y.append(target[i + window_size - 1])
            all_t.append(times[i + window_size - 1])
            all_seq.append(seq_id)

    if not all_X:
        return (
            np.empty((0, window_size, len(cfg.FEATURE_COLUMNS)), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype="datetime64[ns]"),
            np.empty((0,), dtype=object),
        )
    return (
        np.stack(all_X),
        np.array(all_y, dtype=np.float32),
        np.array(all_t, dtype="datetime64[ns]"),
        np.array(all_seq, dtype=object),
    )


def create_sequences():
    df = _load_clean_df()
    n_sequences = df["sequence_id"].nunique()

    print("=" * 70)
    print(f"SPLIT STRATEGY: {'by whole sequence' if n_sequences > 1 else 'chronological (single sequence fallback)'}")
    print("=" * 70)

    if n_sequences > 1:
        if n_sequences == 2:
            # Special case: with only 2 sequences, a clean 3-way whole-
            # sequence split isn't possible. We hold out the SMALLER
            # sequence entirely as the test set (a fully unseen
            # driver/route - strong leakage protection for testing), and
            # carve a chronological tail off the END of the larger
            # sequence to serve as the validation set (used only for
            # early stopping, not for reporting final metrics).
            counts = df.groupby("sequence_id").size().sort_values(ascending=False)
            big_seq, small_seq = counts.index[0], counts.index[1]
            print(f"Only 2 sequences found ({list(counts.index)}) -> hybrid split:")
            print(f"  test  = whole sequence '{small_seq}' ({counts[small_seq]} rows, fully unseen)")
            print(f"  train/val = sequence '{big_seq}' ({counts[big_seq]} rows), "
                  f"chronological tail ({cfg.VAL_FRACTION*100:.0f}%) held out as val")

            big_df = df[df["sequence_id"] == big_seq].sort_values("timestamp").reset_index(drop=True)
            n_big = len(big_df)
            n_val = int(n_big * cfg.VAL_FRACTION)
            train_df = big_df.iloc[:n_big - n_val].reset_index(drop=True)
            val_df = big_df.iloc[n_big - n_val:].reset_index(drop=True)
            test_df = df[df["sequence_id"] == small_seq].reset_index(drop=True)
        else:
            assignment, current, total = _assign_sequences_to_splits(df)
            for split in ("train", "val", "test"):
                pct = 100 * current[split] / total
                print(f"{split:5s}: sequences={assignment[split]}  rows={current[split]} ({pct:.1f}%)")

            train_df = df[df["sequence_id"].isin(assignment["train"])].reset_index(drop=True)
            val_df = df[df["sequence_id"].isin(assignment["val"])].reset_index(drop=True)
            test_df = df[df["sequence_id"].isin(assignment["test"])].reset_index(drop=True)

            if len(assignment["val"]) == 0 or len(assignment["test"]) == 0:
                raise RuntimeError(
                    "Sequence-level split left val or test with zero sequences even "
                    f"with {n_sequences} sequences available. Add a few more sequences "
                    "(ideally not wildly different in size) so each split gets at "
                    "least one whole sequence."
                )
    else:
        # single-sequence fallback: chronological split
        seq_id = df["sequence_id"].iloc[0]
        df = df.sort_values("timestamp").reset_index(drop=True)
        n = len(df)
        n_train = int(n * cfg.TRAIN_FRACTION)
        n_val = int(n * cfg.VAL_FRACTION)
        train_df = df.iloc[:n_train].reset_index(drop=True)
        val_df = df.iloc[n_train:n_train + n_val].reset_index(drop=True)
        test_df = df.iloc[n_train + n_val:].reset_index(drop=True)
        print(f"Only 1 sequence ({seq_id}) found -> chronological split:")
        print(f"  train rows={len(train_df)} ({cfg.TRAIN_FRACTION*100:.0f}%)  "
              f"[{train_df['timestamp'].iloc[0]} -> {train_df['timestamp'].iloc[-1]}]")
        print(f"  val   rows={len(val_df)} ({cfg.VAL_FRACTION*100:.0f}%)  "
              f"[{val_df['timestamp'].iloc[0]} -> {val_df['timestamp'].iloc[-1]}]")
        print(f"  test  rows={len(test_df)} ({100-cfg.TRAIN_FRACTION*100-cfg.VAL_FRACTION*100:.0f}%)  "
              f"[{test_df['timestamp'].iloc[0]} -> {test_df['timestamp'].iloc[-1]}]")

    # --- fit scalers ONLY on training rows -----------------------------------
    scaler = StandardScaler()
    scaler.fit(train_df[cfg.FEATURE_COLUMNS].values)

    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()
    train_df[cfg.FEATURE_COLUMNS] = scaler.transform(train_df[cfg.FEATURE_COLUMNS].values)
    val_df[cfg.FEATURE_COLUMNS] = scaler.transform(val_df[cfg.FEATURE_COLUMNS].values) if len(val_df) else val_df[cfg.FEATURE_COLUMNS]
    test_df[cfg.FEATURE_COLUMNS] = scaler.transform(test_df[cfg.FEATURE_COLUMNS].values) if len(test_df) else test_df[cfg.FEATURE_COLUMNS]

    os.makedirs(cfg.ARTIFACTS_DIR, exist_ok=True)
    scaler_path = os.path.join(cfg.ARTIFACTS_DIR, "feature_scaler.joblib")
    joblib.dump(scaler, scaler_path)
    print(f"\nFitted feature StandardScaler on TRAIN ONLY -> saved to {scaler_path}")

    target_scaler = StandardScaler()
    target_scaler.fit(train_df[[cfg.TARGET_COLUMN]].values)
    target_scaler_path = os.path.join(cfg.ARTIFACTS_DIR, "target_scaler.joblib")
    joblib.dump(target_scaler, target_scaler_path)
    print(f"Fitted target StandardScaler on TRAIN ONLY -> saved to {target_scaler_path}")

    # --- windowing (independently per sequence => no leakage across splits/sequences)
    X_train, y_train, t_train, seq_train = _make_windows(train_df, cfg.WINDOW_SIZE, cfg.WINDOW_STRIDE)
    X_val, y_val, t_val, seq_val = _make_windows(val_df, cfg.WINDOW_SIZE, cfg.WINDOW_STRIDE)
    X_test, y_test, t_test, seq_test = _make_windows(test_df, cfg.WINDOW_SIZE, cfg.WINDOW_STRIDE)

    print("\n" + "=" * 70)
    print("WINDOWED TRAINING DATA")
    print("=" * 70)
    print(f"Window duration      : {cfg.WINDOW_DURATION_S} s ({cfg.WINDOW_SIZE} timesteps)")
    print(f"Window stride        : {cfg.WINDOW_STRIDE} samples ({cfg.WINDOW_STRIDE * cfg.SAMPLE_PERIOD_S:.2f} s)")
    print(f"Number of features   : {len(cfg.FEATURE_COLUMNS)} ({cfg.FEATURE_COLUMNS})")
    print(f"X_train shape        : {X_train.shape}")
    print(f"y_train shape        : {y_train.shape}")
    print(f"X_val shape          : {X_val.shape}")
    print(f"y_val shape          : {y_val.shape}")
    print(f"X_test shape         : {X_test.shape}")
    print(f"y_test shape         : {y_test.shape}")

    os.makedirs(cfg.PROCESSED_DATA_DIR, exist_ok=True)
    np.savez_compressed(
        os.path.join(cfg.PROCESSED_DATA_DIR, "windows.npz"),
        X_train=X_train, y_train=y_train, t_train=t_train, seq_train=seq_train,
        X_val=X_val, y_val=y_val, t_val=t_val, seq_val=seq_val,
        X_test=X_test, y_test=y_test, t_test=t_test, seq_test=seq_test,
    )
    print(f"\nSaved windowed arrays -> {os.path.join(cfg.PROCESSED_DATA_DIR, 'windows.npz')}")

    return X_train, y_train, X_val, y_val, X_test, y_test, scaler


if False:  # Disabled in this source snapshot; call deliberately after adaptation.
    create_sequences()


### `train_model.py`

In [ ]:
# SOURCE SNAPSHOT — train_model.py
# Do not run unchanged. Copy/refactor into the IDR velocity-training work when its input contract is finalized.

"""
Step 5: Main AI model - GRU regressor.

Architecture:
    IMU sequence (batch, 20, 6)
        -> GRU (2 layers, hidden_size=64, dropout=0.2)
        -> take final timestep's hidden state
        -> Dense(64 -> 32) + ReLU
        -> Dense(32 -> 1)
        -> Predicted vehicle speed (batch, 1)

Why GRU over LSTM
------------------
GRU has fewer parameters than LSTM (no separate cell state / no output
gate) while typically matching LSTM performance on short sequences like
ours (20 timesteps / 2 seconds). Given the modest dataset size (~14.8k
training windows) and the CPU-only compatibility requirement, GRU trains
faster and is less prone to overfitting here, so it's the more appropriate
choice for this lightweight prototype stage.

Runs on GPU automatically if available, otherwise CPU (see `device`
below).
"""
import os
import sys
import json
import time

import numpy as np
import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as cfg

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(cfg.RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.RANDOM_SEED)


class GRUSpeedRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        out, h_n = self.gru(x)          # out: (batch, timesteps, hidden)
        last = out[:, -1, :]            # final timestep's hidden state ("now")
        return self.head(last).squeeze(-1)


def _load_windows():
    path = os.path.join(cfg.PROCESSED_DATA_DIR, "windows.npz")
    if not os.path.exists(path):
        from create_sequences import create_sequences
        create_sequences()
    d = np.load(path)
    return d["X_train"], d["y_train"], d["X_val"], d["y_val"], d["X_test"], d["y_test"]


def train_model(
    hidden_size=64,
    num_layers=2,
    dropout=0.2,
    lr=5e-4,
    batch_size=64,
    max_epochs=100,
    patience=12,
):
    X_train, y_train, X_val, y_val, X_test, y_test = _load_windows()
    n_features = X_train.shape[2]

    # Load the target scaler (fit on TRAIN ONLY in create_sequences.py) and
    # scale targets for training stability; metrics are always computed
    # after inverse-transforming back to km/h.
    target_scaler_path = os.path.join(cfg.ARTIFACTS_DIR, "target_scaler.joblib")
    target_scaler = joblib.load(target_scaler_path)
    y_train_s = target_scaler.transform(y_train.reshape(-1, 1)).astype(np.float32).ravel()
    y_val_s = target_scaler.transform(y_val.reshape(-1, 1)).astype(np.float32).ravel()

    print("=" * 70)
    print("MAIN MODEL: GRU")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Input shape: (batch, {X_train.shape[1]} timesteps, {n_features} features)")
    print("Target (vehicle speed) is standardized for training; metrics reported in km/h.")

    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train_s))
    val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val_s))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = GRUSpeedRegressor(n_features, hidden_size, num_layers, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": []}

    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                val_losses.append(criterion(pred, yb).item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch:3d}/{max_epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
                break

    train_time = time.time() - t0
    print(f"\nTotal training time: {train_time:.1f}s")

    if best_state is not None:
        model.load_state_dict(best_state)

    os.makedirs(cfg.MODELS_DIR, exist_ok=True)
    model_path = os.path.join(cfg.MODELS_DIR, "gru_speed_model.pt")
    torch.save({
        "model_state_dict": model.state_dict(),
        "n_features": n_features,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "dropout": dropout,
        "window_size": X_train.shape[1],
    }, model_path)
    print(f"Saved GRU model -> {model_path}")

    os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
    with open(os.path.join(cfg.RESULTS_DIR, "training_history.json"), "w") as f:
        json.dump(history, f, indent=2)

    config_out = {
        "feature_names": cfg.FEATURE_COLUMNS,
        "target_name": cfg.TARGET_COLUMN,
        "sampling_rate_hz": cfg.SAMPLING_RATE_HZ,
        "window_duration_s": cfg.WINDOW_DURATION_S,
        "window_size_timesteps": cfg.WINDOW_SIZE,
        "window_stride": cfg.WINDOW_STRIDE,
        "normalization": "StandardScaler (mean/std), fit on training split only",
        "model": {
            "type": "GRU",
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "dropout": dropout,
            "learning_rate": lr,
            "batch_size": batch_size,
            "max_epochs": max_epochs,
            "patience": patience,
            "epochs_trained": len(history["train_loss"]),
            "best_val_loss_mse": best_val_loss,
            "device_used": str(device),
        },
    }
    with open(os.path.join(cfg.ARTIFACTS_DIR, "config.json"), "w") as f:
        json.dump(config_out, f, indent=2)
    print(f"Saved run configuration -> {os.path.join(cfg.ARTIFACTS_DIR, 'config.json')}")

    return model, history


if False:  # Disabled in this source snapshot; call deliberately after adaptation.
    train_model()


### `evaluate.py`

In [ ]:
# SOURCE SNAPSHOT — evaluate.py
# Do not run unchanged. Copy/refactor into the IDR velocity-training work when its input contract is finalized.

"""
Step 6: Evaluation and required plots.

Evaluates the trained GRU model on the held-out TEST set only, computes
MAE/RMSE/R2/max-abs-error, and generates all required plots under
results/plots/. Works whether the test set is one held-out chronological
slice (single-sequence fallback) or one or more whole held-out sequences
(multi-sequence split).
"""
import os
import sys
import json

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as cfg
from train_model import GRUSpeedRegressor, device


def _load_windows():
    path = os.path.join(cfg.PROCESSED_DATA_DIR, "windows.npz")
    return np.load(path, allow_pickle=True)


def _load_clean_df():
    parquet_path = os.path.join(cfg.PROCESSED_DATA_DIR, "clean_sequence.parquet")
    csv_path = os.path.join(cfg.PROCESSED_DATA_DIR, "clean_sequence.csv")
    if os.path.exists(parquet_path):
        return pd.read_parquet(parquet_path)
    return pd.read_csv(csv_path, parse_dates=["timestamp"])


def _metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
        "max_abs_error": float(np.max(np.abs(y_true - y_pred))),
    }


def evaluate():
    os.makedirs(cfg.PLOTS_DIR, exist_ok=True)

    d = _load_windows()
    X_test, y_test = d["X_test"], d["y_test"]
    t_test, seq_test = d["t_test"], d["seq_test"]

    # --- load GRU model ------------------------------------------------
    ckpt_path = os.path.join(cfg.MODELS_DIR, "gru_speed_model.pt")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = GRUSpeedRegressor(
        ckpt["n_features"], ckpt["hidden_size"], ckpt["num_layers"], ckpt["dropout"]
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    target_scaler = joblib.load(os.path.join(cfg.ARTIFACTS_DIR, "target_scaler.joblib"))

    with torch.no_grad():
        test_pred_scaled = model(torch.from_numpy(X_test).to(device)).cpu().numpy()
    test_pred = target_scaler.inverse_transform(test_pred_scaled.reshape(-1, 1)).ravel()

    gru_metrics = _metrics(y_test, test_pred)

    # skip MAPE-style metric entirely: many test speeds are 0 km/h, so
    # percentage error is undefined/explodes near zero (see task notes).

    print("=" * 70)
    print("TEST SET EVALUATION (GRU)")
    print("=" * 70)
    print(f"Test sequences: {sorted(set(seq_test.tolist())) if len(seq_test) else '(none)'}")
    for k, v in gru_metrics.items():
        print(f"  {k:15s}: {v:.4f}")

        with open(os.path.join(cfg.RESULTS_DIR, "gru_test_metrics.json"), "w") as f:
          json.dump(gru_metrics, f, indent=2)
        np.savez(os.path.join(cfg.RESULTS_DIR, "gru_test_predictions.npz"), y_true=y_test, y_pred=test_pred)

    # Human-readable CSV of every test-set prediction (easy to open in
    # Excel/VS Code, unlike the .npz above).
        pred_csv = pd.DataFrame({
         "sequence_id": seq_test,
         "timestamp": pd.to_datetime(t_test),
         "actual_speed_kmh": y_test,
         "predicted_speed_kmh": test_pred,
         "error_kmh": test_pred - y_test,
    }).sort_values(["sequence_id", "timestamp"]).reset_index(drop=True)
        pred_csv_path = os.path.join(cfg.RESULTS_DIR, "predictions.csv")
        pred_csv.to_csv(pred_csv_path, index=False)
        print(f"Saved human-readable predictions -> {pred_csv_path}")
    # --- combined results table -----------------------------------------
        baseline_metrics_path = os.path.join(cfg.RESULTS_DIR, "baseline_metrics.json")
        combined = {"gru": gru_metrics}
        if os.path.exists(baseline_metrics_path):
            with open(baseline_metrics_path) as f:
                combined["baseline_random_forest"] = json.load(f)["test"]
        with open(os.path.join(cfg.RESULTS_DIR, "final_results.json"), "w") as f:
            json.dump(combined, f, indent=2)

    # =====================================================================
    # PLOTS
    # =====================================================================
    df = _load_clean_df()
    sequence_names = list(df["sequence_id"].unique())
    multi = len(sequence_names) > 1

    # Build a common x-axis (sample index) with sequence-boundary markers,
    # since raw timestamps are not comparable across different recordings.
    df_sorted_parts = []
    boundaries = []
    offset = 0
    for name in sequence_names:
        sub = df[df["sequence_id"] == name].sort_values("timestamp").reset_index(drop=True)
        sub = sub.copy()
        sub["_x"] = np.arange(len(sub)) + offset
        boundaries.append((offset, name))
        offset += len(sub)
        df_sorted_parts.append(sub)
    df_plot = pd.concat(df_sorted_parts, ignore_index=True)

    def _mark_boundaries(ax):
        if multi:
            for pos, name in boundaries[1:]:
                ax.axvline(pos, color="gray", linestyle=":", linewidth=0.8)
            for pos, name in boundaries:
                ax.text(pos, ax.get_ylim()[1], f" {name}", fontsize=7, va="top", rotation=90)

    xlabel = "Sample index (dotted lines = sequence boundaries)" if multi else "Time"
    x_axis = df_plot["_x"] if multi else df_plot["timestamp"]

    # Plot 1: Accelerometer vs time
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(x_axis, df_plot["acc_x"], label="acc_x", linewidth=0.4)
    ax.plot(x_axis, df_plot["acc_y"], label="acc_y", linewidth=0.4)
    ax.plot(x_axis, df_plot["acc_z"], label="acc_z", linewidth=0.4)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Acceleration (m/s^2)")
    ax.set_title("Accelerometer X/Y/Z vs Time" + (f" ({len(sequence_names)} sequences)" if multi else ""))
    ax.legend()
    _mark_boundaries(ax)
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.PLOTS_DIR, "01_accelerometer_vs_time.png"), dpi=120)
    plt.close(fig)

    # Plot 2: Gyroscope vs time
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(x_axis, df_plot["gyro_x"], label="gyro_x (roll)", linewidth=0.4)
    ax.plot(x_axis, df_plot["gyro_y"], label="gyro_y (pitch)", linewidth=0.4)
    ax.plot(x_axis, df_plot["gyro_z"], label="gyro_z (yaw)", linewidth=0.4)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Angular rate (rad/s)")
    ax.set_title("Gyroscope X/Y/Z vs Time" + (f" ({len(sequence_names)} sequences)" if multi else ""))
    ax.legend()
    _mark_boundaries(ax)
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.PLOTS_DIR, "02_gyroscope_vs_time.png"), dpi=120)
    plt.close(fig)

    # Plot 3: Ground-truth vehicle speed vs time
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(x_axis, df_plot["vehicle_speed"], linewidth=0.5, color="darkgreen")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Vehicle speed (km/h)")
    ax.set_title("Ground-truth Vehicle Speed vs Time" + (f" ({len(sequence_names)} sequences)" if multi else " (full sequence)"))
    _mark_boundaries(ax)
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.PLOTS_DIR, "03_groundtruth_speed_vs_time.png"), dpi=120)
    plt.close(fig)

    # Plot 4: Training loss vs validation loss
    with open(os.path.join(cfg.RESULTS_DIR, "training_history.json")) as f:
        history = json.load(f)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history["train_loss"], label="train loss (MSE)")
    ax.plot(history["val_loss"], label="val loss (MSE)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE loss")
    ax.set_title("GRU Training vs Validation Loss")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.PLOTS_DIR, "04_train_val_loss.png"), dpi=120)
    plt.close(fig)

    # Plots 5-7 use the TEST windows directly, ordered by (sequence, time)
    # using the timestamps/sequence ids saved by create_sequences.py - this
    # works correctly regardless of split strategy.
    order = np.lexsort((t_test.astype("datetime64[ns]").astype(np.int64), seq_test))
    y_true_o = y_test[order]
    y_pred_o = test_pred[order]
    seq_o = seq_test[order]

    # x-axis: sample index within the ordered test set, with boundaries
    # between different test sequences marked
    test_x = np.arange(len(y_true_o))
    test_boundaries = []
    if len(seq_o):
        prev = seq_o[0]
        test_boundaries.append((0, prev))
        for i in range(1, len(seq_o)):
            if seq_o[i] != prev:
                test_boundaries.append((i, seq_o[i]))
                prev = seq_o[i]

    def _mark_test_boundaries(ax):
        if multi and len(test_boundaries) > 1:
            for pos, name in test_boundaries[1:]:
                ax.axvline(pos, color="gray", linestyle=":", linewidth=0.8)

    test_xlabel = "Test window index (ordered by sequence, then time)" if multi else "Time (test set)"

    # Plot 5: Ground-truth vs predicted speed over time (MOST IMPORTANT)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(test_x, y_true_o, label="Ground truth", linewidth=1.0)
    ax.plot(test_x, y_pred_o, label="GRU prediction", linewidth=1.0, alpha=0.8)
    ax.set_xlabel(test_xlabel)
    ax.set_ylabel("Vehicle speed (km/h)")
    ax.set_title("Test Set: Ground-truth vs Predicted Vehicle Speed")
    ax.legend()
    _mark_test_boundaries(ax)
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.PLOTS_DIR, "05_groundtruth_vs_predicted_speed.png"), dpi=120)
    plt.close(fig)

    # Plot 6: Prediction error vs time
    error = y_pred_o - y_true_o
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(test_x, error, linewidth=0.7, color="firebrick")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel(test_xlabel)
    ax.set_ylabel("Prediction error (km/h)")
    ax.set_title("Prediction Error vs Time (predicted - actual)")
    _mark_test_boundaries(ax)
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.PLOTS_DIR, "06_prediction_error_vs_time.png"), dpi=120)
    plt.close(fig)

    # Plot 7: Ground-truth vs predicted scatter with y=x line
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_test, test_pred, s=4, alpha=0.4)
    lims = [min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())]
    ax.plot(lims, lims, color="red", linestyle="--", label="y = x (ideal)")
    ax.set_xlabel("Ground-truth speed (km/h)")
    ax.set_ylabel("Predicted speed (km/h)")
    ax.set_title("Ground-truth vs Predicted Speed (Test Set)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.PLOTS_DIR, "07_scatter_groundtruth_vs_predicted.png"), dpi=120)
    plt.close(fig)

    # Bonus: GPS trajectory plot for data verification (GPS exists in S-file)
    from inspect_data import load_all_raw
    all_raw = load_all_raw()
    fig, ax = plt.subplots(figsize=(6, 6))
    for name, (s_raw, _) in all_raw.items():
        ax.plot(s_raw[cfg.S_COLS["gps_lon"]], s_raw[cfg.S_COLS["gps_lat"]], linewidth=0.5, label=name)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("GPS Trajectory (Smartphone) - Data Verification")
    ax.set_aspect("equal", adjustable="datalim")
    if multi:
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(cfg.PLOTS_DIR, "08_gps_trajectory.png"), dpi=120)
    plt.close(fig)

    print(f"\nAll plots saved to {cfg.PLOTS_DIR}")
    return gru_metrics


if False:  # Disabled in this source snapshot; call deliberately after adaptation.
    evaluate()


## Notebook-to-backend handoff

When the deterministic stage is ready, extract only the adapted window builder, model class, training loop, and evaluation metrics into our package. Do not copy the teammate's raw-data assumptions, filesystem globals, or current checkpoint as a runtime dependency.